# Lab 2: SELF-ASSESSMENT: Visualizing Transformer Dataflow

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 2, Dataflow Execution Models  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)  
**Runtime**: Coursera Labs (CPU only — **no GPU needed here**)  
**Expected time**: 15–20 minutes

---

> **Self-Assessment notebook — the 40 points here are ungraded.** This is where
> you check how well your Lab 2 run went. You do not write code or re-trace any
> graph here — you paste the `lab2_results.json` your Colab run produced, and all
> **40 ungraded Self-Assessment points** are checked from it automatically. There
> is no written analysis and no human review.
>
> **The graded item for this lab is the separate Concept Check quiz** (10 points,
> auto-graded in the Coursera Graded Assignment section). It is mandatory and
> requires **80%** to pass. Nothing in *this* notebook affects your course grade.

## Learning objectives
By the end of this lab you will be able to:
1. Trace a transformer block into an FX graph with torch.fx.symbolic_trace.
2. Annotate every node with FLOPs and bytes and compute its arithmetic intensity.
3. Classify nodes as compute-bound / memory-bound against your device's true ridge.
4. Predict which nodes become scheduling bottlenecks on a dataflow accelerator.

## Prerequisites
A completed run of
[Lab2_Transformer_Dataflow_STUDENT.ipynb](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab2_Transformer_Dataflow_STUDENT.ipynb)
on Google Colab, with its **Part 7** cell run so that **`lab2_results.json`**
exists.

## Why this notebook exists

Lab 2 traces a transformer block into an FX graph and annotates every node with
FLOPs and bytes, so you run it on **Google Colab**. This Coursera notebook is
where your results get **checked, for your own feedback**.

This notebook has **no GPU, no model download and no network access**,
so it cannot re-trace your graph. What it *can* do — and what it does — is check
that your numbers are **internally consistent**: the FLOP and byte totals have
to follow from your per-node table, each node's arithmetic intensity from its
own FLOPs and bytes, and both class partitions from those same numbers against
the ridges you used.

Lab 2 has an advantage over Lab 1 here. Lab 1 measured *time*, which varies with
hardware. Lab 2 does **static analysis**: a fixed architecture traced at fixed
dimensions produces a **deterministic** graph. The node counts, the shapes and
the FLOP totals are therefore *exact* — the same on a T4, an A100, or a
CPU-only runtime.

## The workflow

| Step | Where | What you do |
|------|-------|-------------|
| 1 | GitHub → Colab | Open [Lab2_Transformer_Dataflow_STUDENT.ipynb](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab2_Transformer_Dataflow_STUDENT.ipynb), then **File → Save a copy in Drive** |
| 2 | Colab | Complete every TODO and run all cells. **A GPU is optional for this lab** — it is static analysis |
| 3 | Colab | Run **Part 7**, the final cell. It writes **`lab2_results.json`** and prints it inline |
| 4 | Colab | Copy the JSON straight from that cell's output, or open the file browser (📁, left sidebar) and double-click `lab2_results.json` |
| 5 | **Here** | Paste it into the `RESULTS_JSON` cell at the top of this notebook, then **Kernel → Restart & Run All** for your self-assessment |

> **No GPU is required for a full self-assessment score.** Lab 2 is static
> analysis: `symbolic_trace` and `ShapeProp` produce the same graph, the same
> shapes and the same FLOP and byte totals on a T4 or on a CPU-only runtime.
> Every one of the 40 points is reachable either way.
>
> **Students are encouraged to experiment with other GPUs and accelerators to
> expand their learning.** The one hardware-dependent number here is your device
> ridge, and watching the classification move as the ridge changes is the point
> of Part 4. There is **no extra credit** for doing so; all 40 Self-Assessment
> points come from a single free-tier run.

## How this notebook is checked (40 ungraded Self-Assessment points)

| Part | Points | What it checks |
|------|-------:|----------------|
| **A — Graph structure invariants** | 10 | The traced graph: op histogram, node/edge counts, the shapes you read off the figure |
| **B — Cost-model self-consistency** | 18 | Your totals, AI ranking and data movers recomputed from your own per-node FLOPs and bytes |
| **C — Ridge classification & the ridge shift** | 12 | Both partitions recomputed from your numbers; the classification must *flip* when the ridge changes |
| **Ungraded notebook total** | **40** | fully auto-graded from your JSON for Self-Assessment |

**40 points is a full self-assessment score, and a CPU-only Colab run earns all
of it.** Every check is **auto-graded from the JSON you paste** — there is no
written analysis and no human review. Parts A and B (28 points) are exact and
fully hardware-independent; Part C accepts any device ridge from the course's
hardware table, so no hardware is privileged.

Part B carries the most weight because the per-node cost model is the most work
in the lab — and everything in Part C is recomputed from it.

A **score table** near the end of this notebook reports your 40-point
self-assessment total, so you know exactly which parts of the lab went well.

**Graded Quiz points:**  
A 10 point graded Coursera Assignment quiz is given to assess your understanding
from this lab experience. It is **mandatory**, and you need **80%** to pass.

| Quiz | Points | Comes from |
|---|---:|---|
| Concept Check quiz | 10 | Auto-graded Coursera Assignment |

> **Rounding is fine.** Every numeric check allows 2% relative tolerance, which
> is far more than the rounding in the lab's printed output. Transfer the values
> as the lab produced them and you will be well inside it.

> **Do not skip the Concept Check quiz.** It lives in the Coursera **Graded
> Assignment** section, is auto-graded as its own item, and is worth **10 graded
> points** — the only points for this lab that count toward your course grade.
> It is mandatory and requires 80% to pass.


## Use your own measurements

These 40 points are ungraded, so there is nothing to gain by editing numbers —
the value of this notebook is the feedback. Use the graph and cost model from
**your own** Colab run of Lab 2.

Be aware that the checks in Part B are *cross-checks*: they recompute each
derived quantity from your own raw per-node table and require the two to agree.
Because this lab's evidence is a deterministic graph rather than a timing
measurement, an edited number is unusually easy to spot — the totals, the
intensity ranking and both class partitions all have to agree with the same
per-node FLOPs and bytes at once. Transcription and rounding are fine; invented
values are not.

---
## Your results  &nbsp;·&nbsp; *start here*

Paste the **entire contents** of `lab2_results.json` from your Colab run into the
next cell, between the triple quotes — from the opening `{` through the closing `}`.

Part 7 of the lab prints the JSON inline, so you can copy it straight from that
cell's output without opening the file browser.

Keep the raw-string wrapper (the `r` and the triple quotes) exactly as it is: it
means backslashes and quotes inside your JSON cannot break the cell.

```python
RESULTS_JSON = r"""
paste the contents of lab2_results.json here
"""
```

> **This cell only stores your text — it does not check it yet.** After pasting,
> use **Kernel → Restart & Run All**: the Setup cell below loads the helpers, and
> every check in Parts A–C runs against what you pasted here.

> **If you cannot paste**, upload `lab2_results.json` next to this notebook
> instead (Coursera Labs: the **upload** button in the file browser) and leave the
> placeholder text alone. The loader checks the pasted text first, then falls back
> to the file.



In [ ]:
# Paste the contents of lab2_results.json between the triple quotes.
# (Leave the surrounding raw-string triple quotes exactly as they are.)
# This cell only stores the text; the checks run after Kernel -> Restart & Run All.

# your code here
raise NotImplementedError
RESULTS_JSON = r"""
paste the contents of lab2_results.json here
"""


---
## Setup

The next two cells are **read-only — just run them.** The first defines the
loader and the comparison helpers used by every check below; the second loads
the JSON you pasted above and echoes it back so you can confirm it arrived
intact. They import only the Python standard library: nothing here needs
`torch`, a GPU, or the network.



In [ ]:
# === Grading helpers (read-only) =======================================
import json, ast, os, math

TOL = 0.02            # 2% relative tolerance on derived quantities

# Lab 2's TinyTransformerBlock, at the dimensions the lab ships with.
HIDDEN, HEADS, FFN = 512, 8, 2048
BATCH, SEQ         = 2, 128
HEAD_DIM           = HIDDEN // HEADS          # 64
BATCH_SEQ          = BATCH * SEQ              # 256

# FX op kinds symbolic_trace can emit.
FX_OPS = ("placeholder", "call_function", "call_module", "call_method",
          "get_attr", "output")

# The nine nn.Module submodules of the block.
SUBMODULES = ("q_proj", "k_proj", "v_proj", "o_proj", "ln1", "ln2",
              "ffn_gate", "ffn_up", "ffn_down")

# Every ridge in the course's shared HW_SPECS table, plus the two Spyre memory
# systems. Part C accepts any of these, so no hardware is privileged.
#   T4 203 | A100-40 201 | A100-80 153 | H100 295 | Spyre LPDDR5 480 | Spyre SRAM 9.8
RIDGE_MIN, RIDGE_MAX = 100.0, 500.0

REQUIRED_LS_KEYS = {
    "accelerator", "graph_capture_mechanism", "graph_size",
    "compile_time_work", "scheduling_granularity",
    "compute_bound_nodes", "memory_bound_nodes",
}

REQUIRED_READING_KEYS = {
    "qkv_flat_shape", "qkv_head_shape", "head_split_is_free",
    "score_shape", "score_quadratic_dim",
    "widest_tensor_shape", "widest_producer",
}


def rel(a, b):
    """Relative difference between a and b, safe at b == 0."""
    return abs(float(a) - float(b)) / max(abs(float(b)), 1e-12)


def num(x, field="value"):
    """Read x as a float.

    The lab saves JSON with default=str, so a numpy/torch scalar can arrive as
    the string "1048576.0" rather than a number. Coerce instead of failing.
    """
    if isinstance(x, bool):
        raise AssertionError(f"{field}: expected a number, got a boolean.")
    if x is None:
        raise AssertionError(f"{field}: is missing (None). Re-run the Colab lab.")
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        t = x.strip().replace(",", "").replace("_", "")
        for suffix in ("FLOPs/byte", "flops/byte", "bytes", "byte", "FLOPs",
                       "flops", "MB", "mb", "GB", "KB", "ms", "x"):
            if t.endswith(suffix):
                t = t[: -len(suffix)].strip()
        try:
            return float(t)
        except ValueError:
            pass
    raise AssertionError(f"{field}: could not read {x!r} as a number.")


def ints(x, field="shape"):
    """Read a shape as a tuple of ints, from a list, tuple or '(2, 128, 512)'."""
    if isinstance(x, str):
        t = x.strip().strip("()[]")
        parts = [p for p in t.replace(",", " ").split() if p]
        try:
            return tuple(int(float(p)) for p in parts)
        except ValueError:
            raise AssertionError(f"{field}: could not read {x!r} as a shape.")
    if isinstance(x, (list, tuple)):
        return tuple(int(num(v, field)) for v in x)
    raise AssertionError(f"{field}: expected a shape (list of ints), got {x!r}.")


def numel(shape):
    p = 1
    for d in shape:
        p *= int(d)
    return p


def truthy(x, field="value"):
    """Read a JSON/str/py boolean. default=str can turn True into 'True'."""
    if isinstance(x, bool):
        return x
    if isinstance(x, str) and x.strip().lower() in ("true", "false"):
        return x.strip().lower() == "true"
    raise AssertionError(f"{field}: expected True or False, got {x!r}.")


def classify(flops, bytes_, ridge, cheap_bytes):
    """Lab 2's Part 4 classification rule, verbatim."""
    if bytes_ < cheap_bytes:
        return "cheap"
    ai = (flops / bytes_) if bytes_ > 0 else 0.0
    return "compute" if ai >= ridge else "memory"


def _parse(text):
    text = text.strip()
    if not text:
        raise AssertionError("No results provided.")
    try:
        return json.loads(text)                 # normal path
    except Exception:
        pass
    try:
        return ast.literal_eval(text)           # tolerate a pasted Python dict
    except Exception:
        pass
    if text.startswith("{") and not text.endswith("}"):
        raise AssertionError(
            "Your pasted JSON looks incomplete. Copy from the opening '{' "
            "through the final '}'.")
    if any(q in text for q in ("“", "”", "‘", "’")):
        raise AssertionError(
            "Your pasted text contains curly 'smart quotes', which are not "
            "valid JSON. Paste as plain text.")
    raise AssertionError(
        "Could not parse your results. Paste the exact contents of "
        "lab2_results.json. (Curly 'smart quotes' from a word processor "
        "will break JSON — paste as plain text.)")


def load_results(blob=""):
    """Pasted JSON first, then an uploaded lab2_results.json, else explain."""
    data, src = None, None
    if blob and blob.strip() and "paste the contents" not in blob:
        data, src = _parse(blob), "pasted text"
    else:
        for p in ("lab2_results.json", "submission/lab2_results.json",
                  os.path.expanduser("~/lab2_results.json")):
            if os.path.exists(p):
                with open(p) as f:
                    data, src = _parse(f.read()), p
                break
    if data is None:
        raise AssertionError(
            "No results found. Either paste the contents of lab2_results.json "
            "into RESULTS_JSON near the top of this notebook, or upload the "
            "file next to this notebook.")
    if not isinstance(data, dict):
        raise AssertionError("Your results should be a JSON object (a dict).")

    for k in ("op_counts", "n_nodes", "n_edges", "nodes", "total_flops",
              "total_bytes", "class_counts", "landscape_row_w2"):
        if k not in data:
            raise AssertionError(
                f"Key '{k}' is missing. Re-run Part 7 of the Colab lab to "
                f"regenerate lab2_results.json. (If your file has no 'nodes' "
                f"key you are running an older copy of Lab 2 — re-download it.)")

    # --- per-node raw table: the ground truth for everything in Part B ---
    nodes = []
    for e in data["nodes"]:
        if not isinstance(e, dict):
            raise AssertionError("Each entry of 'nodes' should be an object.")
        for k in ("name", "op", "flops", "bytes"):
            if k not in e:
                raise AssertionError(f"A node entry is missing '{k}'.")
        nodes.append({
            "name":  str(e["name"]),
            "op":    str(e["op"]),
            "flops": num(e["flops"], f"nodes['{e['name']}']['flops']"),
            "bytes": num(e["bytes"], f"nodes['{e['name']}']['bytes']"),
            "ai":    num(e["ai"], f"nodes['{e['name']}']['ai']")
                     if e.get("ai") is not None else None,
            "class": str(e["class"]) if e.get("class") is not None else None,
        })
    if not nodes:
        raise AssertionError("'nodes' is empty — did Part 3 of the lab run?")

    op_counts = {str(k): int(num(v, f"op_counts['{k}']"))
                 for k, v in dict(data["op_counts"]).items()}
    class_counts = {str(k): int(num(v, f"class_counts['{k}']"))
                    for k, v in dict(data["class_counts"]).items()}

    out = {
        "op_counts":    op_counts,
        "n_fx_nodes":   int(num(data.get("n_fx_nodes",
                                         sum(op_counts.values())), "n_fx_nodes")),
        "n_nodes":      int(num(data["n_nodes"], "n_nodes")),
        "n_edges":      int(num(data["n_edges"], "n_edges")),
        "nodes":        nodes,
        "total_flops":  num(data["total_flops"], "total_flops"),
        "total_bytes":  num(data["total_bytes"], "total_bytes"),
        "class_counts": class_counts,
        "landscape_row_w2": dict(data["landscape_row_w2"]),
        "dataflow_reading": dict(data.get("dataflow_reading") or {}),
        "top5_ai": [[str(e[0]), num(e[1], "a top5_ai value")]
                    for e in (data.get("top5_ai") or [])],
        "zero_flop_movers": [str(s) for s in (data.get("zero_flop_movers") or [])],
        "cheap_bytes":  num(data.get("cheap_bytes", 1e4), "cheap_bytes"),
        "gpu_ridge_ai": num(data.get("gpu_ridge_ai", 203), "gpu_ridge_ai"),
        "spyre_ridge_lpddr": num(data["spyre_ridge_lpddr"], "spyre_ridge_lpddr")
                             if data.get("spyre_ridge_lpddr") is not None else None,
        "spyre_ridge_scratchpad": num(data["spyre_ridge_scratchpad"],
                                      "spyre_ridge_scratchpad")
                                  if data.get("spyre_ridge_scratchpad") is not None else None,
        "spyre_class_counts": {str(k): int(num(v, "spyre_class_counts"))
                               for k, v in dict(data["spyre_class_counts"]).items()}
                              if data.get("spyre_class_counts") else None,
        "_source": src,
    }
    return out


# --- score bookkeeping (for the courtesy readout near the end) ----------
# Each graded cell calls record(...) as its LAST line, after its asserts. A
# cell that raises never reaches that line, so "absent" means "failed".
_SCORE = {}


def record(gid):
    """Mark a graded cell as passed. Called at the end of each graded cell."""
    _SCORE[gid] = True


# Every autograded cell, with its part and points. The tally walks this, so it
# must stay in step with the cells' nbgrader metadata.
MANIFEST = [
    ('a_op_counts_partition',       "A", 2),
    ('a_submodules_traced',         "A", 2),
    ('a_attention_matmuls',         "A", 1),
    ('a_drawn_graph_shape',         "A", 1),
    ('a_reading_head_split',        "A", 2),
    ('a_reading_blowup',            "A", 2),
    ('b_total_flops',               "B", 3),
    ('b_total_bytes',               "B", 3),
    ('b_ai_per_node',               "B", 4),
    ('b_top5_ranking',              "B", 3),
    ('b_zero_flop_movers',          "B", 3),
    ('b_landscape_row',             "B", 2),
    ('c_class_partition',           "C", 3),
    ('c_ridge_band',                "C", 1),
    ('c_spyre_ridges',              "C", 3),
    ('c_spyre_partition',           "C", 2),
    ('c_ridge_flip',                "C", 3),
]
CHECK_LABELS = {
    'a_op_counts_partition'   : 'op_counts is a valid partition of the FX graph',
    'a_submodules_traced'     : 'all nine block submodules traced',
    'a_attention_matmuls'     : 'both attention matmuls present with FLOPs',
    'a_drawn_graph_shape'     : 'drawn dataflow graph is a plausible DAG',
    'a_reading_head_split'    : 'head split read off the figure',
    'a_reading_blowup'        : 'score blowup and FFN widening read off the figure',
    'b_total_flops'           : 'total_flops is the sum of your per-node FLOPs',
    'b_total_bytes'           : 'total_bytes is the sum of your per-node bytes',
    'b_ai_per_node'           : "every node's AI equals its own flops/bytes",
    'b_top5_ranking'          : 'top5_ai is the true intensity ranking',
    'b_zero_flop_movers'      : 'zero_flop_movers matches your own table',
    'b_landscape_row'         : 'landscape row fields, graph size and class counts',
    'c_class_partition'       : 'class_counts recomputed at your device ridge',
    'c_ridge_band'            : 'device ridge is a real hardware ratio',
    'c_spyre_ridges'          : 'both Spyre ridges and the inequality direction',
    'c_spyre_partition'       : 'spyre_class_counts recomputed on the same graph',
    'c_ridge_flip'            : 'the classification flips when the ridge moves',
}
PART_NAMES = {
    "A": "Graph structure invariants",
    "B": "Cost-model self-consistency",
    "C": "Ridge classification & shift",
}

print("Helpers loaded. Tolerance on derived values: "
      f"{TOL*100:.0f}%   (hidden={HIDDEN}, heads={HEADS}, ffn={FFN}, "
      f"batch={BATCH}, seq={SEQ})")
print(f"Self-Assessment checks: {len(MANIFEST)}   "
      f"Parts A-C total: {sum(p for _, _part, p in MANIFEST)} ungraded points"
      f"   (fully auto-graded, no human review)")


In [ ]:
# === Load and echo your results (read-only) ============================
# Every graded cell below calls load_results() itself, so one bad field can
# never cascade into unrelated failures.
_r = load_results(RESULTS_JSON)

print(f"Loaded from: {_r['_source']}\n")
print("  FX graph")
for _k in FX_OPS:
    if _k in _r["op_counts"]:
        print(f"    {_k:16s} {_r['op_counts'][_k]:4d}")
print(f"    {'TOTAL':16s} {sum(_r['op_counts'].values()):4d} FX nodes")
print(f"\n  Drawn dataflow graph : {_r['n_nodes']} nodes / {_r['n_edges']} edges")
print(f"  Per-node table       : {len(_r['nodes'])} rows")
print(f"\n  total_flops {_r['total_flops']:>18,.0f}  "
      f"({_r['total_flops']/1e9:.3f} GFLOP)")
print(f"  total_bytes {_r['total_bytes']:>18,.0f}  "
      f"({_r['total_bytes']/1e6:.2f} MB)")
print(f"  overall AI  {_r['total_flops']/max(_r['total_bytes'],1):>18.2f}  FLOPs/byte")

print(f"\n  Device ridge            {_r['gpu_ridge_ai']:8.1f} FLOPs/byte")
if _r["spyre_ridge_lpddr"] is not None:
    print(f"  Spyre ridge (LPDDR5)    {_r['spyre_ridge_lpddr']:8.1f} FLOPs/byte")
if _r["spyre_ridge_scratchpad"] is not None:
    print(f"  Spyre ridge (scratchpad){_r['spyre_ridge_scratchpad']:8.1f} FLOPs/byte")

print(f"\n  Classification at the device ridge : {_r['class_counts']}")
if _r["spyre_class_counts"]:
    print(f"  Classification at the scratchpad   : {_r['spyre_class_counts']}")

print("\n  Top 5 nodes by arithmetic intensity:")
for _nm, _ai in _r["top5_ai"][:5]:
    print(f"    {_nm:24s} AI = {_ai:8.2f}")
print(f"\n  Zero-FLOP data movers ({len(_r['zero_flop_movers'])}): "
      f"{', '.join(_r['zero_flop_movers'][:8])}"
      f"{' ...' if len(_r['zero_flop_movers']) > 8 else ''}")
print(f"\n  Accelerator: {_r['landscape_row_w2'].get('accelerator')}")
print(f"  Graph size : {_r['landscape_row_w2'].get('graph_size')}")
print("\nCheck these against your Colab output before continuing.")


---
# Part A — Graph structure invariants (10 ungraded points)

A fixed architecture traced at fixed dimensions produces a **deterministic**
graph. Every check in this Part is therefore *exact* — it holds on a T4, an
A100, or a CPU-only runtime, because `symbolic_trace` and `ShapeProp` do not
consult the device.

In [ ]:
# A1 (2 pts): op_counts must be a valid partition of the traced FX graph.
_r = load_results(RESULTS_JSON)
_oc = _r["op_counts"]
assert _oc, "op_counts is empty. Did TODO 1.3 run?"
_unknown = [k for k in _oc if k not in FX_OPS]
assert not _unknown, (
    f"op_counts has keys torch.fx never emits: {_unknown}. The valid op kinds "
    f"are {list(FX_OPS)}.")
for _k, _v in _oc.items():
    assert _v >= 0, f"op_counts['{_k}'] is negative."
assert _oc.get("placeholder", 0) >= 1, (
    "A traced graph needs at least one placeholder (the input `x`).")
assert _oc.get("output", 0) == 1, (
    f"A traced graph has exactly one output node, found "
    f"{_oc.get('output', 0)}.")
_tot = sum(_oc.values())
assert _tot == len(_r["nodes"]), (
    f"op_counts sums to {_tot} but your per-node table has {len(_r['nodes'])} "
    f"rows. Both describe the same FX graph, so they must agree.")
print(f"[OK] op_counts is a valid partition of {_tot} FX nodes: "
      f"{ {k: _oc[k] for k in FX_OPS if k in _oc} }")
record("a_op_counts_partition")


In [ ]:
# A2 (2 pts): all nine nn.Module submodules of the block must appear as
# call_module nodes. This is what proves you traced TinyTransformerBlock and
# not something else.
_r = load_results(RESULTS_JSON)
_mods = {n["name"] for n in _r["nodes"] if n["op"] == "call_module"}
_missing = [m for m in SUBMODULES if m not in _mods]
assert not _missing, (
    f"These submodules are missing from your call_module nodes: {_missing}.\n"
    f"TinyTransformerBlock has nine: {list(SUBMODULES)}. FX names a "
    f"call_module node after its target, so all nine should appear verbatim.")
assert _r["op_counts"].get("call_module", 0) == 9, (
    f"Expected exactly 9 call_module nodes (one per submodule), found "
    f"{_r['op_counts'].get('call_module', 0)}.")
print(f"[OK] all nine submodules traced: {sorted(_mods)}")
record("a_submodules_traced")


In [ ]:
# A3 (1 pts): the two attention matmuls must be in the graph.
# Q.K^T and attn.V are call_function torch.matmul nodes, and they are the only
# nodes in this block that do matmul FLOPs outside the Linears.
_r = load_results(RESULTS_JSON)
_mm = [n for n in _r["nodes"]
       if n["op"] == "call_function" and n["name"].startswith("matmul")]
assert len(_mm) == 2, (
    f"Expected exactly 2 torch.matmul nodes (Q.K^T and attn.V), found "
    f"{len(_mm)}: {[n['name'] for n in _mm]}.")
for _n in _mm:
    assert _n["flops"] > 0, (
        f"'{_n['name']}' is a matmul but you gave it 0 FLOPs. Use "
        f"matmul_flops() on the two input shapes.")
print(f"[OK] both attention matmuls present with non-zero FLOPs: "
      f"{[n['name'] for n in _mm]}")
record("a_attention_matmuls")


In [ ]:
# A4 (1 pts): the drawn dataflow graph must be a sane DAG.
# The renderer drops the shape-bookkeeping nodes (getattr/getitem) and the
# output node, so it is SMALLER than the full FX graph but describes the same
# dataflow. A connected DAG on n nodes needs at least n-1 edges.
_r = load_results(RESULTS_JSON)
_n, _e = _r["n_nodes"], _r["n_edges"]
assert _n > 0 and _e > 0, "n_nodes and n_edges must both be positive."
assert _n <= len(_r["nodes"]), (
    f"The drawn graph has {_n} nodes but the FX graph only has "
    f"{len(_r['nodes'])}. The renderer filters nodes out, it cannot add any.")
assert _e >= _n - 1, (
    f"{_e} edges across {_n} nodes cannot connect the graph "
    f"(a connected DAG needs at least n-1 = {_n-1}).")
assert _e <= _n * (_n - 1) // 2, (
    f"{_e} edges is impossible on a {_n}-node DAG (max "
    f"{_n*(_n-1)//2}).")
# Every node in a transformer block feeds at least one consumer, and most feed
# exactly one, so the edge/node ratio sits a little above 1.
assert 0.8 <= _e / _n <= 2.5, (
    f"edges/nodes = {_e/_n:.2f} is implausible for a traced transformer block "
    f"(expect roughly 1.0-1.3).")
print(f"[OK] drawn graph is a plausible DAG: {_n} nodes / {_e} edges "
      f"(ratio {_e/_n:.2f})")
record("a_drawn_graph_shape")


In [ ]:
# A5 (2 pts): the head split you read off the figure (TODO 2.4).
# [B,S,H] is reinterpreted as [B,N,S,D] with H == N*D, and because reshape and
# transpose only change strides, the split does no FLOPs.
_r = load_results(RESULTS_JSON)
_dr = _r["dataflow_reading"]
_miss = sorted(REQUIRED_READING_KEYS - set(_dr))
assert not _miss, f"dataflow_reading is missing {_miss}. Complete TODO 2.4."
_flat = ints(_dr["qkv_flat_shape"], "qkv_flat_shape")
_head = ints(_dr["qkv_head_shape"], "qkv_head_shape")
assert len(_flat) == 3, f"qkv_flat_shape should be 3-D [B,S,H], got {_flat}."
assert len(_head) == 4, f"qkv_head_shape should be 4-D [B,N,S,D], got {_head}."
assert _flat == (BATCH, SEQ, HIDDEN), (
    f"qkv_flat_shape should be the shape q_proj produced, "
    f"({BATCH}, {SEQ}, {HIDDEN}), got {_flat}.")
assert _head == (BATCH, HEADS, SEQ, HEAD_DIM), (
    f"qkv_head_shape should be ({BATCH}, {HEADS}, {SEQ}, {HEAD_DIM}) "
    f"[B,N,S,D], got {_head}.")
assert _flat[2] == _head[1] * _head[3], (
    f"hidden {_flat[2]} must equal num_heads * head_dim = "
    f"{_head[1]} * {_head[3]} = {_head[1]*_head[3]}.")
assert numel(_flat) == numel(_head), (
    "a reshape preserves element count — the two shapes must hold the same "
    "number of elements.")
assert truthy(_dr["head_split_is_free"], "head_split_is_free") is True, (
    "reshape and transpose only reinterpret layout — no FLOPs — so the head "
    "split is free in this cost model.")
print(f"[OK] head split read correctly: {_flat} -> {_head} "
      f"(H = {_head[1]}x{_head[3]}), and it is free")
record("a_reading_head_split")


In [ ]:
# A6 (2 pts): the score blowup and the FFN widening (TODO 2.4).
_r = load_results(RESULTS_JSON)
_dr = _r["dataflow_reading"]
_miss = sorted(REQUIRED_READING_KEYS - set(_dr))
assert not _miss, f"dataflow_reading is missing {_miss}. Complete TODO 2.4."

# Q.K^T is [B, N, S, S]: quadratic in sequence length.
_score = ints(_dr["score_shape"], "score_shape")
assert len(_score) == 4, f"score_shape should be 4-D, got {_score}."
assert _score[2] == _score[3], (
    f"the Q.K^T output is [B, N, S, S] — its last two dims must be equal, "
    f"got {_score}.")
assert _score == (BATCH, HEADS, SEQ, SEQ), (
    f"score_shape should be ({BATCH}, {HEADS}, {SEQ}, {SEQ}), got {_score}.")
assert int(num(_dr["score_quadratic_dim"], "score_quadratic_dim")) == _score[3], (
    f"the quadratic dimension is the one appearing twice, {_score[3]} "
    f"(the sequence length).")

# The FFN up-projections produce the widest tensor in the graph.
_wide = ints(_dr["widest_tensor_shape"], "widest_tensor_shape")
assert _wide == (BATCH, SEQ, FFN), (
    f"widest_tensor_shape should be ({BATCH}, {SEQ}, {FFN}) — the FFN "
    f"widening to ffn_dim — got {_wide}.")
assert numel(_wide) > numel(_score), (
    f"the FFN tensor ({numel(_wide):,} elements) should be wider than the "
    f"score tensor ({numel(_score):,}).")
_prod = str(_dr["widest_producer"]).strip()
_names = {n["name"] for n in _r["nodes"]}
assert _prod in _names, f"'{_prod}' is not a node in your graph."
assert _prod in ("ffn_gate", "ffn_up"), (
    f"the widest tensor is first produced by one of the FFN up-projections "
    f"(ffn_gate or ffn_up), not '{_prod}'.")
print(f"[OK] blowup read correctly: scores {_score} (quadratic in {_score[3]}), "
      f"widest {_wide} from '{_prod}'")
record("a_reading_blowup")


---
# Part B — Cost-model self-consistency (18 ungraded points)

Each check below recomputes one of your derived values from your **own** per-node
FLOPs and bytes and requires the two to agree within 2%.

This is the anti-fabrication mechanism, and it is the heaviest Part because the
per-node cost model was the heaviest work in the lab. Your per-node table, your
totals, your intensity ranking and your class partitions all describe the same
graph, so a value that was edited by hand generally fails several of these at
once. Transcription and rounding are fine.

In [ ]:
# B1 (3 pts): total_flops must be the sum of your own per-node FLOPs.
_r = load_results(RESULTS_JSON)
_sum = sum(n["flops"] for n in _r["nodes"])
_e = rel(_r["total_flops"], _sum)
assert _e <= TOL, (
    f"total_flops = {_r['total_flops']:,.0f} but your per-node FLOPs sum to "
    f"{_sum:,.0f}, off by {_e*100:.1f}%. The total is the sum over all nodes.")
assert _sum > 0, "Your per-node FLOPs are all zero — did TODO 3.1 run?"
print(f"[OK] total_flops consistent with the per-node table "
      f"({_sum:,.0f} FLOPs, {_e*100:.2f}% difference)")
record("b_total_flops")


In [ ]:
# B2 (3 pts): total_bytes must be the sum of your own per-node bytes.
_r = load_results(RESULTS_JSON)
_sum = sum(n["bytes"] for n in _r["nodes"])
_e = rel(_r["total_bytes"], _sum)
assert _e <= TOL, (
    f"total_bytes = {_r['total_bytes']:,.0f} but your per-node bytes sum to "
    f"{_sum:,.0f}, off by {_e*100:.1f}%.")
assert _sum > 0, "Your per-node bytes are all zero — did TODO 3.1 run?"
print(f"[OK] total_bytes consistent with the per-node table "
      f"({_sum:,.0f} bytes, {_e*100:.2f}% difference)")
record("b_total_bytes")


In [ ]:
# B3 (4 pts): every node's arithmetic intensity must equal its own
# flops / bytes (and 0.0 where bytes == 0). This is TODO 3.2a.
_r = load_results(RESULTS_JSON)
_have = [n for n in _r["nodes"] if n["ai"] is not None]
assert len(_have) == len(_r["nodes"]), (
    f"{len(_r['nodes']) - len(_have)} node(s) have no 'ai' value. TODO 3.2a "
    f"asks for node.meta['ai'] on every node.")
_bad = []
for _n in _r["nodes"]:
    _want = (_n["flops"] / _n["bytes"]) if _n["bytes"] > 0 else 0.0
    if rel(_n["ai"], _want) > TOL and abs(_n["ai"] - _want) > 1e-6:
        _bad.append((_n["name"], _n["ai"], _want))
assert not _bad, (
    "These nodes' arithmetic intensity does not match their own flops/bytes:\n"
    + "\n".join(f"  {nm}: ai = {got:.4f} but flops/bytes = {want:.4f}"
                for nm, got, want in _bad[:6])
    + ("\n  ..." if len(_bad) > 6 else "")
    + "\nAI = flops / bytes, guarding against bytes == 0.")
print(f"[OK] arithmetic intensity self-consistent on all "
      f"{len(_r['nodes'])} nodes")
record("b_ai_per_node")


In [ ]:
# B4 (3 pts): top5_ai must really be the five highest-intensity nodes in your
# own table, highest first (TODO 3.2b).
_r = load_results(RESULTS_JSON)
_t5 = _r["top5_ai"]
assert len(_t5) == 5, f"top5_ai must hold exactly 5 entries, found {len(_t5)}."
_ais = [a for _, a in _t5]
assert _ais == sorted(_ais, reverse=True), (
    f"top5_ai must be sorted highest-first, got {[round(a,2) for a in _ais]}.")

_by_name = {n["name"]: n for n in _r["nodes"]}
for _nm, _ai in _t5:
    assert _nm in _by_name, f"'{_nm}' in top5_ai is not a node in your graph."
    _n = _by_name[_nm]
    assert _n["bytes"] > 0, (
        f"'{_nm}' has bytes == 0, so it has no meaningful intensity. "
        f"TODO 3.2b says to include only nodes with bytes > 0.")
    _want = _n["flops"] / _n["bytes"]
    assert rel(_ai, _want) <= TOL, (
        f"top5_ai lists {_nm} at AI = {_ai:.2f} but its own flops/bytes give "
        f"{_want:.2f}.")

# No node outside the list may beat the fifth entry.
_cut = _ais[4]
_better = [n["name"] for n in _r["nodes"]
           if n["bytes"] > 0 and n["name"] not in {nm for nm, _ in _t5}
           and n["flops"] / n["bytes"] > _cut * (1 + TOL)]
assert not _better, (
    f"These nodes have higher intensity than your fifth entry "
    f"({_cut:.2f}) but are missing from top5_ai: {_better[:5]}.")
print(f"[OK] top5_ai is the true ranking. Highest: {_t5[0][0]} at "
      f"AI = {_t5[0][1]:.2f}")
record("b_top5_ranking")


In [ ]:
# B5 (3 pts): zero_flop_movers must be exactly the nodes with bytes > 0 and
# flops == 0 — the pure data-movement ops a dataflow compiler wants to fuse
# away (TODO 3.2c).
_r = load_results(RESULTS_JSON)
_got  = set(_r["zero_flop_movers"])
_want = {n["name"] for n in _r["nodes"]
         if n["bytes"] > 0 and n["flops"] == 0}
assert _got, "zero_flop_movers is empty. Complete TODO 3.2c."
_extra   = sorted(_got - _want)
_missing = sorted(_want - _got)
assert not _missing and not _extra, (
    f"zero_flop_movers does not match your own table.\n"
    f"  missing: {_missing[:8]}\n"
    f"  unexpected: {_extra[:8]}\n"
    f"The set is every node with bytes > 0 AND flops == 0.")
# These are the reshape/transpose/softmax/elementwise ops, never the Linears.
_linears = [nm for nm in _got if nm in SUBMODULES and nm.endswith("_proj")]
assert not _linears, (
    f"{_linears} are Linear projections — they do real FLOPs and cannot be "
    f"zero-FLOP movers. Check your Part 3.1 annotation.")
assert any(nm.startswith("transpose") for nm in _got), (
    "the transposes move bytes and compute nothing — they belong in this set.")
print(f"[OK] zero_flop_movers matches your table exactly "
      f"({len(_got)} pure data movers)")
record("b_zero_flop_movers")


In [ ]:
# B6 (2 pts): the Week-2 landscape row must be complete and must quote your
# own numbers. Three things in one cell, because they are one deliverable:
#   (a) all 7 fields present and filled,
#   (b) graph_size quotes your own node and edge counts,
#   (c) the bound-node counts mirror class_counts.
_r = load_results(RESULTS_JSON)
_ls = _r["landscape_row_w2"]

# (a) structure
_got = set(_ls)
assert _got == REQUIRED_LS_KEYS, (
    f"landscape_row_w2 keys are wrong.\n"
    f"  missing: {sorted(REQUIRED_LS_KEYS - _got)}\n"
    f"  unexpected: {sorted(_got - REQUIRED_LS_KEYS)}")
for _k, _v in _ls.items():
    assert _v is not None, f"landscape_row_w2['{_k}'] is still None."
for _k in ("accelerator", "graph_capture_mechanism", "graph_size",
           "compile_time_work", "scheduling_granularity"):
    assert isinstance(_ls[_k], str) and _ls[_k].strip(), (
        f"landscape_row_w2['{_k}'] should be a short non-empty description.")

# (b) graph_size must mention both of your own counts
_found, _tok = [], ""
for _ch in str(_ls["graph_size"]):
    if _ch.isdigit():
        _tok += _ch
    else:
        if _tok:
            _found.append(int(_tok))
        _tok = ""
if _tok:
    _found.append(int(_tok))
assert _r["n_nodes"] in _found and _r["n_edges"] in _found, (
    f"graph_size = {_ls['graph_size']!r} should quote both of your own counts, "
    f"e.g. '{_r['n_nodes']} nodes / {_r['n_edges']} edges'.")

# (c) bound-node counts mirror class_counts
_cc = _r["class_counts"]
_lc = int(num(_ls["compute_bound_nodes"], "compute_bound_nodes"))
_lm = int(num(_ls["memory_bound_nodes"], "memory_bound_nodes"))
assert _lc == _cc.get("compute", -1), (
    f"landscape_row_w2['compute_bound_nodes'] = {_lc} but "
    f"class_counts['compute'] = {_cc.get('compute')}.")
assert _lm == _cc.get("memory", -1), (
    f"landscape_row_w2['memory_bound_nodes'] = {_lm} but "
    f"class_counts['memory'] = {_cc.get('memory')}.")

print(f"[OK] landscape row complete, quotes your own graph size "
      f"({_ls['graph_size']!r}), and mirrors class_counts "
      f"(compute {_lc}, memory {_lm})")
record("b_landscape_row")


---
# Part C — Ridge classification and the ridge shift (12 ungraded points)

Part 4 of the lab classified every node against your device's ridge, then
re-classified the *same graph* against Spyre's on-chip scratchpad ridge.

These checks recompute both partitions from your own per-node numbers. The
device ridge itself is the one hardware-dependent value in this lab, so it is
accepted anywhere in the range the course's hardware table covers (T4 203,
A100 153–201, H100 295) — no device is privileged.

The last check is the important one: the classification must **flip** when the
ridge changes while the graph stays fixed. That is the evidence you understood
the ridge is a property of the *hardware*, not the model.

In [ ]:
# C1 (3 pts): class_counts must be a valid partition recomputed from your own
# per-node numbers against your own ridge (TODO 4.1).
_r = load_results(RESULTS_JSON)
_cc = _r["class_counts"]
assert set(_cc) == {"compute", "memory", "cheap"}, (
    f"class_counts keys should be exactly compute/memory/cheap, got "
    f"{sorted(_cc)}.")
assert sum(_cc.values()) == len(_r["nodes"]), (
    f"class_counts sums to {sum(_cc.values())} but there are "
    f"{len(_r['nodes'])} nodes. Every node must be classified exactly once.")

_want = {"compute": 0, "memory": 0, "cheap": 0}
for _n in _r["nodes"]:
    _want[classify(_n["flops"], _n["bytes"],
                   _r["gpu_ridge_ai"], _r["cheap_bytes"])] += 1
assert _cc == _want, (
    f"class_counts = {_cc} but applying the Part 4.1 rule to your own "
    f"per-node table at ridge {_r['gpu_ridge_ai']:.1f} gives {_want}.\n"
    f"The rule: cheap if bytes < {_r['cheap_bytes']:g}, else compute if "
    f"ai >= ridge, else memory.")
print(f"[OK] class_counts recomputed from your own table: {_cc}")
record("c_class_partition")


In [ ]:
# C2 (1 pts): the device ridge must be a real ratio from the course's hardware
# table. ridge = peak_fp16_tflops / peak_bandwidth_tb_s.
_r = load_results(RESULTS_JSON)
_rp = _r["gpu_ridge_ai"]
assert RIDGE_MIN <= _rp <= RIDGE_MAX, (
    f"gpu_ridge_ai = {_rp:.1f} is outside the range covered by the course's "
    f"HW_SPECS table ({RIDGE_MIN:.0f}-{RIDGE_MAX:.0f}). The entries are "
    f"T4 203, A100-40 201, A100-80 153, H100 295. A CPU-only run falls back "
    f"to the T4 row, which is correct.")
print(f"[OK] device ridge = {_rp:.1f} FLOPs/byte is a valid table entry")
record("c_ridge_band")


In [ ]:
# C3 (3 pts): both Spyre ridges, and the direction of the inequality (4.3a/b).
# More bandwidth for the same compute means a LOWER ridge.
_r = load_results(RESULTS_JSON)
_lp, _sp = _r["spyre_ridge_lpddr"], _r["spyre_ridge_scratchpad"]
assert _lp is not None and _sp is not None, (
    "Both Spyre ridges are missing. Complete TODO 4.3a and 4.3b, then re-run "
    "Part 7.")
# SPYRE_SPECS: 98.0 TFLOPS over 0.204 TB/s LPDDR5 and 10.0 TB/s scratchpad.
_want_lp, _want_sp = 98.0 / 0.204, 98.0 / 10.0
assert rel(_lp, _want_lp) <= TOL, (
    f"spyre_ridge_lpddr = {_lp:.1f} but SPYRE_SPECS gives "
    f"fp16_tflops / lpddr5_tb_s = 98.0 / 0.204 = {_want_lp:.1f}.")
assert rel(_sp, _want_sp) <= TOL, (
    f"spyre_ridge_scratchpad = {_sp:.1f} but SPYRE_SPECS gives "
    f"fp16_tflops / scratchpad_tb_s = 98.0 / 10.0 = {_want_sp:.1f}.")
assert _sp < _lp, (
    f"the scratchpad ridge ({_sp:.1f}) must be BELOW the LPDDR5 ridge "
    f"({_lp:.1f}): same compute over ~49x the bandwidth gives a lower ridge.")
print(f"[OK] Spyre ridges correct: LPDDR5 {_lp:.1f}, scratchpad {_sp:.1f} "
      f"FLOPs/byte ({_lp/_sp:.0f}x apart)")
record("c_spyre_ridges")


In [ ]:
# C4 (2 pts): spyre_class_counts must be the SAME graph re-classified against
# the scratchpad ridge (TODO 4.3c) — not a fresh set of numbers.
_r = load_results(RESULTS_JSON)
_sc = _r["spyre_class_counts"]
assert _sc is not None, (
    "spyre_class_counts is missing. Complete TODO 4.3c and re-run Part 7.")
assert set(_sc) == {"compute", "memory", "cheap"}, (
    f"spyre_class_counts keys should be compute/memory/cheap, got "
    f"{sorted(_sc)}.")
assert sum(_sc.values()) == len(_r["nodes"]), (
    f"spyre_class_counts sums to {sum(_sc.values())} but there are "
    f"{len(_r['nodes'])} nodes — the graph did not change, so every node must "
    f"still be classified exactly once.")

_want = {"compute": 0, "memory": 0, "cheap": 0}
for _n in _r["nodes"]:
    _want[classify(_n["flops"], _n["bytes"],
                   _r["spyre_ridge_scratchpad"], _r["cheap_bytes"])] += 1
assert _sc == _want, (
    f"spyre_class_counts = {_sc} but re-classifying your own per-node table "
    f"at the scratchpad ridge {_r['spyre_ridge_scratchpad']:.1f} gives "
    f"{_want}. Use the same rule as 4.1, only the ridge changes.")
# The cheap set is a byte threshold, so it cannot move when only the ridge does.
assert _sc["cheap"] == _r["class_counts"]["cheap"], (
    f"the 'cheap' count is a bytes < {_r['cheap_bytes']:g} threshold and does "
    f"not depend on the ridge, so it must be unchanged "
    f"({_r['class_counts']['cheap']}), got {_sc['cheap']}.")
print(f"[OK] spyre_class_counts recomputed from the same table: {_sc}")
record("c_spyre_partition")


In [ ]:
# C5 (3 pts): the punchline. The graph is identical; only the ridge moved.
# A LOWER ridge can only promote memory-bound nodes to compute-bound, never
# demote them — and on this block it must promote at least one.
_r = load_results(RESULTS_JSON)
_cc, _sc = _r["class_counts"], _r["spyre_class_counts"]
assert _sc is not None, "Complete TODO 4.3c and re-run Part 7."
assert _r["spyre_ridge_scratchpad"] < _r["gpu_ridge_ai"], (
    f"the scratchpad ridge ({_r['spyre_ridge_scratchpad']:.1f}) should be "
    f"below your device ridge ({_r['gpu_ridge_ai']:.1f}) — that is why the "
    f"classification moves.")
assert _sc["compute"] >= _cc["compute"], (
    f"compute-bound went from {_cc['compute']} to {_sc['compute']}. A lower "
    f"ridge cannot REDUCE the compute-bound count: every node that already "
    f"cleared the higher bar still clears the lower one.")
assert _sc["memory"] <= _cc["memory"], (
    f"memory-bound went from {_cc['memory']} to {_sc['memory']}. A lower "
    f"ridge cannot increase it.")
_moved = _sc["compute"] - _cc["compute"]
assert _moved > 0, (
    f"no node changed class. At a scratchpad ridge of "
    f"{_r['spyre_ridge_scratchpad']:.1f} FLOPs/byte the GEMMs in this block "
    f"(AI 128-158) should clear the bar — check your 4.3c re-classification.")
# Conservation: what left 'memory' is what arrived in 'compute'.
assert _cc["memory"] - _sc["memory"] == _moved, (
    "nodes can only move between 'memory' and 'compute' here, so the two "
    "deltas must match.")
print(f"[OK] ridge shift confirmed: {_r['gpu_ridge_ai']:.0f} -> "
      f"{_r['spyre_ridge_scratchpad']:.1f} FLOPs/byte moved {_moved} node(s) "
      f"from memory-bound into compute-bound on an unchanged graph")
record("c_ridge_flip")


---
## Your self-assessment score

The cell below summarizes the checks above so you can see which parts of the lab
went well. It is read-only, and these 40 points are **ungraded** — nothing here
affects your course grade.



In [ ]:
# === Your self-assessment score (read-only readout -- nothing here is graded) ===
# Run this after running every cell above (Kernel -> Restart & Run All).
# It summarizes the checks above so you can see which parts of the lab went
# submit. It cannot change your grade.
try:
    _rows, _earned, _possible = [], {}, {}
    for _gid, _part, _pts in MANIFEST:
        _ok = _gid in _SCORE
        _earned[_part]   = _earned.get(_part, 0)   + (_pts if _ok else 0)
        _possible[_part] = _possible.get(_part, 0) + _pts
        _rows.append((_part, _gid, _pts, _ok))

    _parts  = [p for p in ("A", "B", "C") if p in _possible]
    _auto_e = sum(_earned[p]   for p in _parts)
    _auto_p = sum(_possible[p] for p in _parts)

    print("=" * 68)
    print("  YOUR SELF-ASSESSMENT SCORE  (40 ungraded points)")
    print("=" * 68)
    for _p in _parts:
        _bar = "#" * int(round(18 * _earned[_p] / max(_possible[_p], 1)))
        print(f"  {_p} - {PART_NAMES[_p]:28s} {_earned[_p]:3d} / {_possible[_p]:<3d} {_bar}")
    print(f"  {'-' * 64}")
    print(f"  TOTAL                            {_auto_e:3d} / {_auto_p:<3d}")

    # Per-check detail: every graded cell, what it checks, and the points it
    # earned. This is the full breakdown behind the part totals above.
    print("=" * 68)
    print("  DETAIL - every check, in order")
    print("=" * 68)
    _n = {}
    for _p, _g, _pt, _ok in _rows:
        _n[_p] = _n.get(_p, 0) + 1
        _tag   = f"{_p}{_n[_p]}"
        _mark  = "PASS" if _ok else "FAIL"
        _got   = _pt if _ok else 0
        _what  = CHECK_LABELS.get(_g, _g)
        print(f"  {_tag:<4s} {_mark}  {_got}/{_pt}  {_what}")
    print(f"  {'-' * 64}")
    print(f"  TOTAL      {_auto_e:2d}/{_auto_p:<2d}")

    _failed = [(p, g) for p, g, _, ok in _rows if not ok]
    if _failed:
        print("-" * 68)
        print("  Cells not yet passing - scroll up to their error messages:")
        for _p, _g in _failed:
            print(f"    [{_p}] {_g}  -  {CHECK_LABELS.get(_g, '')}")
        print("  Each is independent: fixing one cannot cost you another.")
    else:
        print("-" * 68)
        print("  Every autograded cell passed.")

    print("=" * 68)
    print("  Wrapping up")
    print("  1. Kernel -> Restart & Run All, then re-read this table. If you")
    print("     ran cells out of order, the numbers above can be stale.")
    print("  2. These 40 points are UNGRADED self-assessment feedback, auto-")
    print("     checked from your pasted JSON. No write-up, no human review,")
    print("     and nothing here affects your course grade.")
    print("  3. The GRADED item is the Concept Check quiz: a separate auto-")
    print("     graded Coursera Assignment worth 10 points. It is mandatory")
    print("     and needs 80% to pass. Do not skip it.")
    print("=" * 68)
except Exception as _e:
    # A courtesy readout must never be the reason a passing student panics.
    print(f"[score readout unavailable: {type(_e).__name__}: {_e}]")
    print("Your check cells above are unaffected - each prints [OK] when it passes.")


---
## Wrapping up

1. **Run every cell top to bottom** (Kernel → Restart & Run All). Each check
   prints `[OK]` when it passes, and the **score table** just above reports your
   40-point self-assessment total.
2. Any cell that raises an error scores 0 **for that cell only** — the checks are
   independent, so fix what you can; the rest of your self-assessment still counts.
3. Confirm the echo table under your pasted JSON matches your Colab output.
4. **Take the Concept Check quiz.** It is a separate auto-graded Coursera
   **Graded Assignment** worth **10 graded points**, it is mandatory, and you
   need **80%** to pass. The 40 points in this notebook are ungraded feedback;
   the quiz is what counts toward your grade.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `Could not parse your results` | Paste as plain text; curly "smart quotes" break JSON. Copy from `{` through `}`. |
| `Your pasted JSON looks incomplete` | You copied only part of the output. Select all of it. |
| `Key 'nodes' is missing` | Your `lab2_results.json` came from an older copy of Lab 2. Re-download the student notebook and re-run **Part 7**. |
| `op_counts sums to N but your per-node table has M rows` | The two came from different runs. Re-run the whole Colab notebook top to bottom, then Part 7. |
| `arithmetic intensity does not match flops/bytes` | TODO 3.2a must be `flops / bytes`, with `0.0` where `bytes == 0`. |
| `zero_flop_movers does not match your own table` | It is every node with `bytes > 0` **and** `flops == 0` — no Linears, no matmuls. |
| `no node changed class` | TODO 4.3c must re-classify against `spyre_ridge_scratchpad`, not the device ridge. |
| `class_counts ... gives {...}` | Apply the Part 4.1 rule exactly: cheap if `bytes < CHEAP_BYTES`, else compute if `ai >= ridge`, else memory. |
| `gpu_ridge_ai is outside the range` | Use `ridge = fp16_tflops / bw_tb_s` from `HW_SPECS`. A CPU-only run correctly falls back to the T4 row. |
| `graph_size ... should quote both of your own counts` | Build it from your own values, e.g. `f"{G.number_of_nodes()} nodes / {n_edges} edges"`. |

Your Colab notebook stays yours — nothing from it is uploaded except the values
you pasted here.

---

*Course: Dataflow Architectures for AI, Week 2 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*